#Background

You are given daily production and equipment sensor data from several oil wells. Your task is to clean the data, engineer useful features, train a simple model, and explain your decisions.

The dataset has these columns:      

```
well_id      
date       
oil_bbl       
gas_mcf        
water_bbl       
tubing_pressure       
casing_pressure       
choke_size       
runtime_hours       
failure_next_7d    
```   

failure_next_7d is the target label:       
1 means the well had an equipment-related failure within the next 7 days.       
0 means no failure.        

##Part 1 — Data Cleaning and Basic Python, 10 minutes

Write a function:

Requirements:

1. Convert date to datetime.             
2. Sort by well_id and date.
3. Remove duplicate rows based on well_id and date, keeping the latest row.
4. Replace negative values in production/sensor columns with NaN.
5. Fill missing numeric values using forward fill within each well_id.
6. Drop rows where oil_bbl is still missing after forward fill.

Production/sensor columns:
```
["oil_bbl", "gas_mcf", "water_bbl", "tubing_pressure",
 "casing_pressure", "choke_size", "runtime_hours"]
 ```

In [0]:
import pandas as pd
import numpy as np

In [0]:
def clean_production_data(df):
    """
    Input: raw pandas DataFrame
    Output: cleaned pandas DataFrame
    """
    df = df.copy()
    # convert date to datetime
    df['date'] = pd.to_datetime(df['date'])
    
    # sort
    df = df.sort_values(by = ['well_id','date'])
    
    # remove duplicate
    df = df.drop_duplicates(subset = ['well_id','date'], keep = 'last')
    
    # replace negative values
    sensor_columns = ["oil_bbl", "gas_mcf", "water_bbl", "tubing_pressure",
 "casing_pressure", "choke_size", "runtime_hours"]
    
    df[sensor_columns] = df[sensor_columns].mask(
        df[sensor_columns] < 0,
        np.nan
    )
    # df.loc[:, sensor_columns][df[sensor_columns] < 0] = np.nan
    
    # forward fill
    df[sensor_columns] = df.groupby("well_id")[sensor_columns].ffill()
    
    # drop nan
    df = df.dropna(subset = ['oil_bbl'])

    return df


##Part 2 — Feature Engineering, 15 minutes

Write a function:

Add these features for each well_id:

1. __oil_7d_avg__: rolling 7-day average of oil_bbl
2. __oil_7d_std__: rolling 7-day standard deviation of oil_bbl
3. __water_cut__: water_bbl / (oil_bbl + water_bbl)
4. __gor__: gas_mcf / oil_bbl
5. __pressure_diff__: casing_pressure - tubing_pressure
6. __oil_decline_7d__: current oil_bbl minus oil production 7 days ago
7. __runtime_3d_avg__: rolling 3-day average of runtime_hours

Important: avoid target leakage.

In [0]:
def add_features(df):
    """
    Input: cleaned DataFrame
    Output: DataFrame with new ML features
    """
    df = df.copy()

    grouped = df.groupby("well_id")
    df["oil_7d_avg"] = grouped["oil_bbl"].transform(lambda x: x.rolling(7, min_periods = 1).mean())
    df["oil_7d_std"] = grouped["oil_bbl"].transform(lambda x: x.rolling(7, min_periods = 1).std())

    df['water_cut'] = df['water_bbl'] / (df['oil_bbl'] + df['water_bbl'])
    df['gor'] = df['gas_mcf'] / df['oil_bbl']
    df['pressure_diff'] = df['casing_pressure'] - df['tubing_pressure']
    df['oil_decline_7d'] = grouped['oil_bbl'].diff(7)
    df['runtime_3d_avg'] = grouped['runtime_hours'].transform(lambda x: x.rolling(3, min_periods = 1).mean())

    return df


##Part 3 — Modeling, 20 minutes

Write a function:

Requirements:

1. Use these features:
```
[
    "oil_7d_avg",
    "oil_7d_std",
    "water_cut",
    "gor",
    "pressure_diff",
    "oil_decline_7d",
    "runtime_3d_avg",
    "choke_size"
]
```
2. Split data by time, not random split:
- training data: first 80% of dates
- test data: last 20% of dates
3. Train a RandomForestClassifier.
4. Return evaluation metrics:
- accuracy
- precision
- recall
- F1 score
- ROC AUC
5. Handle missing feature values properly.

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def train_failure_model(df):
    """
    Input: feature DataFrame
    Output: trained model, evaluation dictionary
    """
    features = [
    "oil_7d_avg",
    "oil_7d_std",
    "water_cut",
    "gor",
    "pressure_diff",
    "oil_decline_7d",
    "runtime_3d_avg",
    "choke_size"
    ]

    df = df.sort_values(by = 'date')

    split = int(len(df)*0.8)
    X_train = df.iloc[:split][features].fillna(0)
    y_train = df.iloc[:split]['failure_next_7d'].fillna(0)

    X_test = df.iloc[split:][features].fillna(0)
    y_test = df.iloc[split:]['failure_next_7d'].fillna(0)

    model = RandomForestClassifier(n_estimators = 100, random_state = 42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:,1]

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_probs)
    }
    return model, metrics

##Part 4 — Debugging, 10 minutes

The following code has at least four problems. Identify and fix them.

```
def bad_model(df):
    df["date"] = pd.to_datetime(df["date"])

    features = ["oil_bbl", "gas_mcf", "water_bbl", "failure_next_7d"]

    X = df[features]
    y = df["failure_next_7d"]

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, random_state=42
    )

    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    return accuracy_score(y_test, preds)
```
Write your corrected version and briefly explain the bugs.

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
def bad_model(df):
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(by = ['date'])

    # features = ["oil_bbl", "gas_mcf", "water_bbl", "failure_next_7d"]
    features = ["oil_bbl", "gas_mcf", "water_bbl"]

    X = df[features]
    y = df["failure_next_7d"]

    # from sklearn.model_selection import train_test_split
    # X_train, X_test, y_train, y_test = train_test_split(
    #     X, y, random_state=42
    # )
    split = int(len(df) * 0.8)

    train = df.iloc[:split]
    test = df.iloc[split:]

    X_train = train[features].fillna(0)
    y_train = train["failure_next_7d"]

    X_test = test[features].fillna(0)
    y_test = test["failure_next_7d"]

    
    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    return accuracy_score(y_test, preds)

##Part 5 — Short Answer, 5 minutes

Answer briefly:

1. Why is random train/test split risky for production forecasting or failure prediction?
2. What is target leakage? Give one oil-and-gas example.
3. If recall is low but accuracy is high, what does that mean for failure prediction?
4. Why might you choose precision-recall AUC instead of ROC AUC?
5. What data quality issues are common in field sensor data?

1. __Why is random split dangerous?__

    Answer:

    Random splitting allows future information to leak into training.

    Example:

    Jan
    Feb
    Mar
    Apr

    Random split may train on April and test on February.

    Impossible in reality.

2. __What is target leakage?__

    Using information unavailable at prediction time.

    Oil example:

    Predicting pump failure using tomorrow's pressure.

3. __High accuracy but low recall__

    Suppose:

    1000 wells

    10 failures

    Predict:

    No failures.

    Accuracy:

    990/1000 = 99%

    Recall:

    0%

    The model is useless.

4. __Why Precision-Recall AUC?__

    Failures are rare.

    Example:

    99% healthy.

    ROC AUC can still look very good.

    PR AUC focuses on positive class.

5. __Common sensor data issues__

    Good answers:

    Missing values
    Duplicate records
    Sensor drift
    Sensor calibration problems
    Outliers
    Clock synchronization errors
    Unit conversion problems
    Communication outages
    Manual data entry mistakes
    Negative production values

In [0]:
sample_data = [
{
"well_id": "W001",
"date": "2025-01-01",
"oil_bbl": 120,
"gas_mcf": 600,
"water_bbl": 30,
"tubing_pressure": 900,
"casing_pressure": 1100,
"choke_size": 24,
"runtime_hours": 24,
"failure_next_7d": 0
},
{
"well_id": "W001",
"date": "2025-01-02",
"oil_bbl": 118,
"gas_mcf": 590,
"water_bbl": 31,
"tubing_pressure": 895,
"casing_pressure": 1090,
"choke_size": 24,
"runtime_hours": 24,
"failure_next_7d": 0
},
{
"well_id": "W001",
"date": "2025-01-03",
"oil_bbl": -10, # bad value
"gas_mcf": 585,
"water_bbl": 32,
"tubing_pressure": 890,
"casing_pressure": 1080,
"choke_size": 24,
"runtime_hours": 24,
"failure_next_7d": 1
},
{
"well_id": "W001",
"date": "2025-01-04",
"oil_bbl": 115,
"gas_mcf": None, # missing value
"water_bbl": 33,
"tubing_pressure": 885,
"casing_pressure": 1075,
"choke_size": 24,
"runtime_hours": 23,
"failure_next_7d": 1
},
{
"well_id": "W001",
"date": "2025-01-04", # duplicate date
"oil_bbl": 116,
"gas_mcf": 580,
"water_bbl": 33,
"tubing_pressure": 886,
"casing_pressure": 1076,
"choke_size": 24,
"runtime_hours": 23,
"failure_next_7d": 1
},
{
"well_id": "W002",
"date": "2025-01-01",
"oil_bbl": 250,
"gas_mcf": 1200,
"water_bbl": 40,
"tubing_pressure": 950,
"casing_pressure": 1180,
"choke_size": 28,
"runtime_hours": 24,
"failure_next_7d": 0
},
{
"well_id": "W002",
"date": "2025-01-02",
"oil_bbl": 248,
"gas_mcf": 1190,
"water_bbl": 42,
"tubing_pressure": 945,
"casing_pressure": 1175,
"choke_size": 28,
"runtime_hours": 24,
"failure_next_7d": 0
},
{
"well_id": "W002",
"date": "2025-01-03",
"oil_bbl": 245,
"gas_mcf": 1180,
"water_bbl": 43,
"tubing_pressure": 940,
"casing_pressure": 1170,
"choke_size": 28,
"runtime_hours": -1, # bad value
"failure_next_7d": 0
},
{
"well_id": "W002",
"date": "2025-01-04",
"oil_bbl": None, # missing value
"gas_mcf": 1170,
"water_bbl": 45,
"tubing_pressure": 935,
"casing_pressure": 1165,
"choke_size": 28,
"runtime_hours": 24,
"failure_next_7d": 1
},
{
"well_id": "W002",
"date": "2025-01-05",
"oil_bbl": 240,
"gas_mcf": 1160,
"water_bbl": 47,
"tubing_pressure": 930,
"casing_pressure": 1160,
"choke_size": 28,
"runtime_hours": 24,
"failure_next_7d": 1
}
]

df = pd.DataFrame(sample_data)

print(df.head())

In [0]:
df_clean = clean_production_data(df)
df_clean.head()

In [0]:
df_feature = add_features(df_clean)
df_feature.display()

In [0]:
model, metrics = train_failure_model(df_feature)
print(metrics)